# Kaggle ASTGCN PEMS04 训练 notebook

本 notebook 面向 Kaggle/Ubuntu 环境，也兼容本地运行。它会自动定位项目根目录和 PEMS04 数据文件，安装当前包，读取 `configs/pems04.yaml`，并提供短训练与正式训练两种模式。

当前仓库的完整 ASTGCN 训练入口仍在建设中，因此这里在 notebook 内定义一个轻量三分支时序模型：分别处理 recent、daily、weekly 片段，并学习三类分支在每个节点和预测步上的融合权重。该流程用于跑通数据、训练、评估和可视化闭环。

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

def find_project_root():
    candidates = [Path.cwd(), Path('/kaggle/working/ASTGCN'), Path('/kaggle/working')]
    for start in candidates:
        if not start.exists():
            continue
        for path in [start, *start.parents]:
            if (path / 'pyproject.toml').exists() and (path / 'configs' / 'pems04.yaml').exists():
                return path.resolve()
    for path in Path('/kaggle/working').rglob('pems04.yaml') if Path('/kaggle/working').exists() else []:
        if path.parent.name == 'configs' and (path.parents[1] / 'pyproject.toml').exists():
            return path.parents[1].resolve()
    raise FileNotFoundError('未找到项目根目录，请确认 pyproject.toml 和 configs/pems04.yaml 已上传。')

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

print('PROJECT_ROOT =', PROJECT_ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT)], check=True)


In [ ]:
import json
import math
import random
import shutil
from copy import deepcopy

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import yaml
import matplotlib.pyplot as plt

from astgcn.data.dataloader import build_dataloaders
from astgcn.utils import ensure_dir, set_random_seed, select_device

print('torch =', torch.__version__)
print('cuda available =', torch.cuda.is_available())


## 运行模式

- `quick`：用于 Kaggle 调试，减少样本量、batch 和 epoch。
- `full`：使用配置文件中的正式训练参数。

In [ ]:
RUN_MODE = 'quick'  # 可改为 'full'

with open(PROJECT_ROOT / 'configs' / 'pems04.yaml', 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

def find_data_file(filename):
    local_path = PROJECT_ROOT / 'data' / 'raw' / 'PEMS04' / filename
    if local_path.exists():
        return local_path.resolve()
    for base in [Path('/kaggle/input'), Path('/kaggle/working'), PROJECT_ROOT]:
        if not base.exists():
            continue
        matches = list(base.rglob(filename))
        if matches:
            return matches[0].resolve()
    raise FileNotFoundError(f'未找到 {filename}，请确认 PEMS04 数据已挂载。')

data_path = find_data_file('pems04.npz')
distance_path = find_data_file('distance.csv')
cfg['dataset']['data_path'] = str(data_path)
cfg['dataset']['distance_path'] = str(distance_path)

if RUN_MODE == 'quick':
    cfg['train']['epochs'] = 2
    cfg['train']['batch_size'] = 16
    cfg['train']['learning_rate'] = 1e-3
    cfg['train']['early_stop_patience'] = 2
    MAX_TRAIN_BATCHES = 20
    MAX_EVAL_BATCHES = 8
else:
    MAX_TRAIN_BATCHES = None
    MAX_EVAL_BATCHES = None

for key in ['save_dir', 'checkpoint_dir', 'log_dir', 'prediction_dir']:
    cfg['log'][key] = str(PROJECT_ROOT / cfg['log'][key]) if not Path(cfg['log'][key]).is_absolute() else cfg['log'][key]

for key in ['save_dir', 'checkpoint_dir', 'log_dir', 'prediction_dir']:
    ensure_dir(cfg['log'][key])
figure_dir = ensure_dir(PROJECT_ROOT / 'outputs' / 'figures')

print('RUN_MODE =', RUN_MODE)
print('data_path =', cfg['dataset']['data_path'])
print('distance_path =', cfg['dataset']['distance_path'])


In [ ]:
set_random_seed(int(cfg['train']['seed']))
device = select_device(cfg['train']['device'])

tw = cfg['time_window']
ds = cfg['dataset']
tr = cfg['train']
sp = cfg['split']

train_loader, val_loader, test_loader, scaler = build_dataloaders(
    data_path=cfg['dataset']['data_path'],
    num_recent=tw['recent_len'],
    num_days=tw['daily_days'],
    num_weeks=tw['weekly_weeks'],
    pred_len=tw['pred_len'],
    batch_size=tr['batch_size'],
    points_per_day=ds['points_per_day'],
    target_dim=ds['target_dim'],
    train_ratio=sp['train_ratio'],
    val_ratio=sp['val_ratio'],
    num_workers=tr['num_workers'],
)

batch = next(iter(train_loader))
for name, value in batch.items():
    print(name, tuple(value.shape))
print('device =', device)


## 三分支融合模型

输入保持数据集输出格式 `[B, T, N, F]`，每个分支独立预测 `[B, N, Tp]`，最后用可学习的 recent/daily/weekly 权重融合。

In [ ]:
class TemporalBranch(nn.Module):
    def __init__(self, input_steps, input_dim, pred_len, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_steps * input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, pred_len),
        )

    def forward(self, x):
        # x: [B, T, N, F] -> [B, N, T * F]
        b, t, n, f = x.shape
        x = x.permute(0, 2, 1, 3).reshape(b, n, t * f)
        return self.net(x)

class ThreeBranchFusionModel(nn.Module):
    def __init__(self, num_nodes, input_dim, pred_len, recent_steps, daily_steps, weekly_steps, hidden_dim=64):
        super().__init__()
        self.recent = TemporalBranch(recent_steps, input_dim, pred_len, hidden_dim)
        self.daily = TemporalBranch(daily_steps, input_dim, pred_len, hidden_dim)
        self.weekly = TemporalBranch(weekly_steps, input_dim, pred_len, hidden_dim)
        self.fusion_logits = nn.Parameter(torch.zeros(3, num_nodes, pred_len))

    def forward(self, recent, daily, weekly, return_parts=False):
        recent_pred = self.recent(recent)
        daily_pred = self.daily(daily)
        weekly_pred = self.weekly(weekly)
        weights = torch.softmax(self.fusion_logits, dim=0)
        stacked = torch.stack([recent_pred, daily_pred, weekly_pred], dim=0)
        final = (weights[:, None, :, :] * stacked).sum(dim=0)
        if return_parts:
            return {
                'final': final,
                'recent': recent_pred,
                'daily': daily_pred,
                'weekly': weekly_pred,
                'weights': weights,
            }
        return final

model = ThreeBranchFusionModel(
    num_nodes=ds['num_nodes'],
    input_dim=ds['input_dim'],
    pred_len=tw['pred_len'],
    recent_steps=tw['recent_len'],
    daily_steps=tw['daily_days'] * tw['pred_len'],
    weekly_steps=tw['weekly_weeks'] * tw['pred_len'],
    hidden_dim=cfg['model']['hidden_channels'],
).to(device)

print(model)


In [ ]:
def move_batch(batch, device):
    return {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

def masked_mape_np(y_true, y_pred, eps=1e-5):
    mask = np.abs(y_true) > eps
    if not np.any(mask):
        return float('nan')
    return float(np.mean(np.abs((y_pred[mask] - y_true[mask]) / y_true[mask])) * 100)

def metrics_np(y_true, y_pred):
    mae = float(np.mean(np.abs(y_pred - y_true)))
    rmse = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))
    mape = masked_mape_np(y_true, y_pred)
    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

def evaluate(model, loader, scaler, max_batches=None):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for i, batch in enumerate(loader):
            if max_batches is not None and i >= max_batches:
                break
            batch = move_batch(batch, device)
            pred = model(batch['recent'].float(), batch['daily'].float(), batch['weekly'].float())
            preds.append(pred.cpu().numpy())
            targets.append(batch['target'].cpu().numpy())
    pred_norm = np.concatenate(preds, axis=0)
    target_norm = np.concatenate(targets, axis=0)
    pred = scaler.inverse_transform_target(pred_norm, target_dim=ds['target_dim'])
    target = scaler.inverse_transform_target(target_norm, target_dim=ds['target_dim'])
    return metrics_np(target, pred), pred, target

optimizer = torch.optim.Adam(model.parameters(), lr=tr['learning_rate'], weight_decay=tr['weight_decay'])
criterion = nn.L1Loss()
best_val = float('inf')
best_path = Path(cfg['log']['checkpoint_dir']) / 'kaggle_astgcn_best.pt'
history = []

for epoch in range(1, tr['epochs'] + 1):
    model.train()
    losses = []
    for i, batch in enumerate(train_loader):
        if MAX_TRAIN_BATCHES is not None and i >= MAX_TRAIN_BATCHES:
            break
        batch = move_batch(batch, device)
        optimizer.zero_grad(set_to_none=True)
        pred = model(batch['recent'].float(), batch['daily'].float(), batch['weekly'].float())
        loss = criterion(pred, batch['target'].float())
        loss.backward()
        if tr.get('grad_clip'):
            torch.nn.utils.clip_grad_norm_(model.parameters(), float(tr['grad_clip']))
        optimizer.step()
        losses.append(float(loss.item()))
    val_metrics, _, _ = evaluate(model, val_loader, scaler, max_batches=MAX_EVAL_BATCHES)
    row = {'epoch': epoch, 'train_loss': float(np.mean(losses)), **{f'val_{k}': v for k, v in val_metrics.items()}}
    history.append(row)
    print(row)
    if val_metrics['MAE'] < best_val:
        best_val = val_metrics['MAE']
        torch.save({'model_state_dict': model.state_dict(), 'config': cfg, 'history': history}, best_path)

print('best checkpoint =', best_path)


In [ ]:
checkpoint = torch.load(best_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

test_metrics, final_pred, target = evaluate(model, test_loader, scaler, max_batches=MAX_EVAL_BATCHES)
print('test metrics:', json.dumps(test_metrics, ensure_ascii=False, indent=2))

pred_path = Path(cfg['log']['prediction_dir']) / 'kaggle_astgcn_predictions.npz'
np.savez_compressed(pred_path, final=final_pred, target=target, history=np.array(history, dtype=object))
print('saved predictions =', pred_path)


## 可视化 target/final/recent/daily/weekly 与融合权重

In [ ]:
model.eval()
sample = move_batch(next(iter(test_loader)), device)
with torch.no_grad():
    parts = model(sample['recent'].float(), sample['daily'].float(), sample['weekly'].float(), return_parts=True)

plot_data = {}
for key in ['final', 'recent', 'daily', 'weekly']:
    plot_data[key] = scaler.inverse_transform_target(parts[key].cpu().numpy(), target_dim=ds['target_dim'])
plot_data['target'] = scaler.inverse_transform_target(sample['target'].cpu().numpy(), target_dim=ds['target_dim'])
weights = parts['weights'].detach().cpu().numpy()

node_id = 0
sample_id = 0
steps = np.arange(tw['pred_len'])
plt.figure(figsize=(10, 5))
for key, style in [('target', 'k-o'), ('final', 'r-o'), ('recent', 'C0--'), ('daily', 'C1--'), ('weekly', 'C2--')]:
    plt.plot(steps, plot_data[key][sample_id, node_id], style, label=key)
plt.xlabel('prediction step')
plt.ylabel('traffic flow')
plt.title(f'PEMS04 node {node_id}: target/final/recent/daily/weekly')
plt.legend()
plt.grid(alpha=0.3)
curve_path = figure_dir / 'kaggle_prediction_components.png'
plt.savefig(curve_path, dpi=160, bbox_inches='tight')
plt.show()

plt.figure(figsize=(9, 4))
plt.plot(steps, weights[0, node_id], label='recent')
plt.plot(steps, weights[1, node_id], label='daily')
plt.plot(steps, weights[2, node_id], label='weekly')
plt.ylim(0, 1)
plt.xlabel('prediction step')
plt.ylabel('fusion weight')
plt.title(f'Fusion weights for node {node_id}')
plt.legend()
plt.grid(alpha=0.3)
weight_path = figure_dir / 'kaggle_fusion_weights.png'
plt.savefig(weight_path, dpi=160, bbox_inches='tight')
plt.show()

print('saved figures =', curve_path, weight_path)
